# The 1D Ideal MHD Equations

\begin{equation}\tag{1}
\mathbf{U}_t + \mathbf{F}(\mathbf{U})_x = 0
\end{equation}


\begin{equation}\tag{2}
\mathbf{U} =
\begin{pmatrix}
\rho \\
\rho u_x \\
\rho u_y \\
\rho u_z \\
E \\
B_x \\
B_y \\
B_z \\
\end{pmatrix},
\qquad
\mathbf{F}(\mathbf{U}) =
\begin{pmatrix}
\rho u_x \\
\rho u_x^2 + p + \frac{1}{2}\|\mathbf{B}\|^2 - B_x^2 \\
\rho u_x u_y - B_x B_y \\
\rho u_x u_z - B_x B_z \\
u_x \left(E + p + \frac{1}{2}\|\mathbf{B}\|^2 \right) - B_x (\mathbf{u}\cdot\mathbf{B}) \\
0 \\
u_x B_y - u_y B_x \\
u_x B_z - u_z B_x 
\end{pmatrix}
\end{equation}

The **conserved variables** are:

- mass density: $\rho(x,t)$  
- momentum density: $\rho \textbf{u} (x,t)  = (\rho u_x, \rho u_y, \rho u_z)$  
- total energy density: $E(x,t)$ 
- magnetic field: $\textbf{B}(x,t) = (B_x, B_y, B_z)$  

The **primitive (physical) variables** are:

- density: $\rho(x,t)$  
- velocity: $\textbf{u}(x,t) = (u_x, u_y, u_z)$  
- pressure: $p(x,t)$ 
- magnetic field: $\textbf{B}(x,t) = (B_x, B_y, B_z)$  

The total energy density is:

\begin{equation}\tag{3}
E = \frac{p}{\gamma - 1} + \frac{1}{2} \rho \|\mathbf{u}\|^2  + \frac{1}{2} \|\mathbf{B}\|^2
\end{equation}

where $\gamma$ is the adiabatic index. Note that $E$ includes both kinetic energy $\frac{1}{2}\rho\|\mathbf{u}\|^2$ and magnetic energy $\frac{1}{2}\|\mathbf{B}\|^2$.

# 1. Why use a finite volume method for MHD at all?


You see equation (1) has the same form as the homogenous advection equation, where F(U) is the flux of the conserved variables. In the advection equations, the conserved quantity is linear with constant speed, aU. The MHD equations are a system of nonlinear conservation laws. This is saying that the time derivative of the solution plus the spatial derivative of the flux is 0. Rearranging equation (1) so that $U_t = -F(U)_x$, then integrating wrt space over a control volume [xL, xR], we see the integral form of the conservation law:

\begin{align}
\frac{d}{dt} \int_{xL}^{xR} U dx &= - \int_{xL}^{xR} F(U)_x dx \nonumber \\
&= F(U(xL,t)) - F(U(xR,t)) \tag{4}
\end{align}

Which just says that the rate of change of the total amount of U inside an interval is equal to the flux entering from the left minus the flux leaving to the right.

The FVM splits the spatial domain into cells and represents the solution U as the cell-average over each cell, $[x_{i-1/2}, x_{i+1/2}]$. The average over each cell $\bar U$ is given by
\begin{equation}
\bar U = \frac{1}{\delta x}\int_{x_{i-1/2}}^{x_{i+1/2}} U(x,t) dx. \tag{5}
\end{equation}

Then, we see that equation (4) becomes
\begin{equation}
\frac{d}{dt} \left( \delta x \bar U \right) = F(U(x_{i-1/2},t)) - F(U(x_{i+1/2},t)) \tag{6}
\end{equation}

Since $\frac{d}{dt} \bar U \approx \frac{\bar U^{n+1} -  \bar U^{n}}{\delta t}$, we have the equation for the updated solution, $U^{n+1}$,

\begin{align}
\bar U^{n+1} &= \bar U^{n} - \frac{\delta t}{\delta x}\left(F(U(x_{i+1/2},t)) - F(U(x_{i-1/2},t))\right) \nonumber \\
&\equiv \bar U^{n} - \frac{\delta t}{\delta x}\left(F_{i+1/2} - F_{i-1/2}\right)
\tag{7}
\end{align}

Then the true fluxes are replaced with a numerical flux which is computed from the Riemann solver which uses cell averages rather than the true point value at the interfaces. The same number $F_{i+1/2}$ is subtracted from cell $i$ and added to cell $i+1$. This means that the flux leaving one cell, enters the next. Therefore, nothing is created or destroyed at the interfaces. Summing over all interior cells, all the fluxes cancel, implying discrete conservation. Thus, the FVM naturally encodes numerical conservation.



# 2\. Explain the process for the 1 dimensional case
First, the conservative form of the solution, 
$$q = \begin{bmatrix} \rho \\
\rho u_x \\
\rho u_y \\
\rho u_z \\
E \\
B_x \\
B_y \\
B_z \\ \end{bmatrix}, $$

is seeded with an initial condition determined by the problem at hand, where $q$ is found as cell averages. The main evolution loop implements the desired boundary conditions (periodic, outflow, etc.) then solves equation (7) repeatedly until the final desired time is reached. Note that in 1D MHD, all fields vary only in x, so $\partial/\partial y = \partial / \partial z = 0$. The divergence-free constraint $$\nabla \cdot B = \partial_x B_x + \partial_y B_y + \partial_z B_z = 0$$ therefore reduces to $\partial_x B_x = 0$. This is automatically satisfied because F[5] = 0 in the flux vector, meaning $\partial B_x/\partial t = 0$, so $B_x$ is constant in both space and time, set entirely by the initial condition.

The timestep, $\delta t$ is chosen in obeyance with the CFL condition: $|v| \leq \frac{\delta x}{\delta t} $ where $CFL \equiv |v| \frac{\delta t}{\delta x}$ must be $\leq 1$. In words, the CFL condition says that the speed of the numerical scheme must be faster than the speed of the solution. Thus, $\delta t$ is chosen in proportion to the $CFL$, $\delta x $ and the speed of the fastest wave in the domain: $$\delta t = CFL \cdot \delta x / |v|,$$
where |v| is found by calculating the 7 different wave speeds admitted by the ideal MHD equations called the entropy, divergence, left/right Alfven, and the fast and slow magnetosonic waves.

Solving equation (7) requires approximations for the fluxes $F_{i+1/2}, F_{i-1/2}$. These approximations are found with a Riemann solver, which uses the values of $q$ to the left $q_L$ and to the right $q_R$ of an interface $i+1/2$, to approximate the flux at that interface. Thus, before implementing a Riemann solver, we must reconstruct the values of $q$ at the interface which uses the slope within each cell to extrapolate the value of q from the cell center out to the cell interface. This can be done with either a piecewise constant, piecewise linear, or piecewise parabolic reconstruction method (PCM, PLM, PPM). 

Once $q_L$ and $q_R$ have been found at the interface, we use either the HLL or HLLD Riemann solvers. The Riemann solver essentially determines which wave the solution is currently being described by: is the solution in the entropy wave region? The divergence wave region? left or right Alfven? fast or slow magnetosonic? In practice, with an HLL or HLLD Riemann solver, the answer to this question is only approximate rather than exact. HLL will choose between three regions determined by the fast magnetosonic wave speed, and is more diffusive, and has a harder time catch discontinuities. The HLLD solver (D stands for discontinuities) chooses between 6 different regions determined by the fast magnetosonic wave speed and a discontinuity wave.

Now, with the flux at interface $i+1/2$ found, we can update equation (7) and restart the process.

The full algorithm goes like:

1. Fill the conserved solution vector with an initial guess $q_0$. 
2. Fill $q_n$ with the appropriate boundary conditions via the ghost cells (necessary to calculate derivatives at the edges of the domain). 
3. Determine the time step $\delta t$ with the spatial mesh size $\delta x$, the chosen $CFL$ number, and the fastest wave in the domain.
4. Reconstruct the values of $q$ at the left and right of each interface $i+1/2$, finding vectors $q_L$ and $q_R$. 
5. Give $q_L$ and $q_R$ to a Riemann solver, which will calculate the fluxes to the left and right of the interfaces then determine which one or what combination of each should be used on that interface.
6. Update the solution from $q_n$ to $q_{n+1}$ with equation (7).
7. Increase the current time $t$ to $t + \delta t$.
8. Repeat steps (2) - (7) until final time is reached.



# The 2D Ideal MHD Equations

\begin{equation}\tag{8}
\mathbf{U}_t + \mathbf{F}(\mathbf{U})_x + \mathbf{F}(\mathbf{U})_y = 0
\end{equation}


\begin{equation}\tag{9}
\mathbf{U} =
\begin{pmatrix}
\rho \\
\rho u_x \\
\rho u_y \\
\rho u_z \\
E \\
B_x \\
B_y \\
B_z \\
\end{pmatrix},
\qquad
\mathbf{F}(\mathbf{U})_x =
\begin{pmatrix}
\rho u_x \\
\rho u_x^2 + p + \frac{1}{2}\|\mathbf{B}\|^2 - B_x^2 \\
\rho u_x u_y - B_x B_y \\
\rho u_x u_z - B_x B_z \\
u_x \left(E + p + \frac{1}{2}\|\mathbf{B}\|^2 \right) - B_x (\mathbf{u}\cdot\mathbf{B}) \\
0 \\
u_x B_y - u_y B_x \\
u_x B_z - u_z B_x \\
\end{pmatrix}
\qquad
\mathbf{F}(\mathbf{U})_y =
\begin{pmatrix}
\rho u_y \\
\rho u_x u_y - B_x B_y \\
\rho u_y^2 + p + \frac{1}{2}\|\mathbf{B}\|^2 - B_y^2 \\
\rho u_y u_z - B_y B_z \\
u_y \left(E + p + \frac{1}{2}\|\mathbf{B}\|^2 \right) - B_y (\mathbf{u}\cdot\mathbf{B}) \\
u_y B_x - u_x B_y \\
0 \\
u_y B_z - u_z B_y 
\end{pmatrix}
\end{equation}

The **conserved variables** are:

- mass density: $\rho(x,y,t)$  
- momentum density: $\rho \textbf{u} (x,y,t)  = (\rho u_x, \rho u_y, \rho u_z)$  
- total energy density: $E(x,y,t)$ 
- magnetic field: $\textbf{B}(x,y,t) = (B_x, B_y, B_z)$  

The **primitive (physical) variables** are:

- density: $\rho(x,y,t)$  
- velocity: $\textbf{u}(x,y,t) = (u_x, u_y, u_z)$  
- pressure: $p(x,y,t)$ 
- magnetic field: $\textbf{B}(x,y,t) = (B_x, B_y, B_z)$  

The total energy density is:

\begin{equation}\tag{10}
E = \frac{p}{\gamma - 1} + \frac{1}{2} \rho \|\mathbf{u}\|^2  + \frac{1}{2} \|\mathbf{B}\|^2
\end{equation}

where $\gamma$ is the adiabatic index. Note that $E$ includes both kinetic energy $\frac{1}{2}\rho\|\mathbf{u}\|^2$ and magnetic energy $\frac{1}{2}\|\mathbf{B}\|^2$.

In the x-momentum flux, the term $\frac{1}{2}\|\mathbf{B}\|^2 - B_x^2$ combines isotropic magnetic pressure ($\frac{1}{2}\|\mathbf{B}\|^2$) with a negative magnetic tension contribution ($-B_x^2$) that resists bending of field lines.



# 3\. Explain the process for the 2 dimensional case
The generic FVM with CTU (or any general updating scheme) does not preserve the $\nabla \cdot B = 0$ constraint imposed by the Maxwell equations. So, we can't update the solution vector $q$ via the 'naive' updating method using the half step x and y fluxes,
$$
\mathcal{U}^{n+1}_{i,j} = \mathcal{U}^n_{i,j} + \frac{\delta t}{\delta x} \left(F^{n+1/2}_{i-1/2,j} - F^{n+1/2}_{i+1/2,j}\right) + \frac{\delta t}{\delta y}\left(G^{n+1/2}_{i,j-1/2} - G^{n+1/2}_{i,j+1/2}\right)
$$

Instead, constrained transport updates $B$ according to the induction law of Maxwell's equations,
\begin{equation} \tag{11}
\frac{\partial B}{\partial t} = - \nabla \times \mathcal{E}.
\end{equation}
Equation (11) naturally ensures that $\nabla \cdot B$ is constant in time since,
\begin{align}
\frac{\partial}{\partial t} \left( \nabla \cdot B \right) &= - \nabla \cdot \left( \nabla \times \mathcal{E}\right) \nonumber \\
&= 0 \nonumber \\
\text{(The divergence of the curl} & \text{ is algebraically 0 for any vector)}. \nonumber
\end{align}
Thus, if we update $B$ with the EMF according to equation (11), and our initial condition enforces $\nabla \cdot B = 0$, the algorithm will implicitly conserve the $\nabla \cdot B = 0$ constraint for all time since the value of $\nabla \cdot B$ is constant in time.

Functionally, this is done by calculating the z component of the EMF at the corners of the cell and then using that to advance the face-valued $B_x$ and $B_y$. This ensures that the same corner EMF updates both neighboring Bx and By face values consistently, which guarantees that $\nabla \cdot B = 0$ is maintained.